# Visual Descriptiveness by Genre

Map agreed annotation labels from `agreed.parquet` to genres via the batch JSON files,
then compute mean ± 2 std of visual descriptiveness per genre.

In [1]:
import json
import pandas as pd
from utils import DATA_DIR
import matplotlib.pyplot as plt

In [2]:
agreed = pd.read_parquet(DATA_DIR / "datasets/small/agreed.parquet")

batch_files = [
    DATA_DIR / "to-annotate/batch_001.json",
    DATA_DIR / "to-annotate/batch_002.json",
]
records = []
for path in batch_files:
    with open(path) as f:
        records.extend(json.load(f))

batches = pd.DataFrame(records)[["text", "genre"]].drop_duplicates(subset="text")

print(f"agreed rows: {len(agreed)}")
print(f"batch rows (deduplicated): {len(batches)}")

agreed rows: 753
batch rows (deduplicated): 1000


In [3]:
df = agreed.merge(batches, on="text", how="inner")

n_unmatched = len(agreed) - len(df)
print(f"Matched: {len(df)} / {len(agreed)}  (unmatched: {n_unmatched})")
print(df["genre"].value_counts().to_string())

Matched: 753 / 753  (unmatched: 0)
genre
Fiction            200
Mystery            144
Fantasy            127
Science Fiction    126
Western            104
History             38
Travel              14


In [4]:
stats = (
    df.groupby("genre")["label"]
    .agg(mean="mean", std="std", count="count")
    .sort_values("mean", ascending=False)
)
stats["lower"] = stats["mean"] - 2 * stats["std"]
stats["upper"] = stats["mean"] + 2 * stats["std"]

display(stats.round(3))

,mean,std,count,lower,upper
genre,,,,,
Fantasy,2.197,1.303,127,-0.410,4.804
Science Fiction,2.151,1.357,126,-0.563,4.865
Travel,2.071,1.592,14,-1.112,5.254
Western,1.962,1.350,104,-0.739,4.662
Fiction,1.845,1.382,200,-0.919,4.609
Mystery,1.715,1.341,144,-0.968,4.398
History,1.579,1.266,38,-0.952,4.110


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

genres = stats.index.tolist()
means = stats["mean"].values
errs = 2 * stats["std"].values

ax.barh(genres, means, xerr=errs, capsize=5, color="steelblue", alpha=0.8)
ax.set_xlabel("Visual Descriptiveness (mean ± 2 std)")
ax.set_title("Visual Descriptiveness by Genre")
ax.set_xlim(0, 5)
ax.axvline(means.mean(), color="red", linestyle="--", linewidth=1, label="overall mean")
ax.legend()

for i, (m, n) in enumerate(zip(means, stats["count"].values)):
    ax.text(0.05, i, f"n={n}", va="center", fontsize=8, color="white")

plt.tight_layout()
fig.savefig(DATA_DIR / "figures" / "genre_descriptiveness.pdf", bbox_inches="tight")
plt.show()